# release16 sparse slice fit root-cause diagnostic

This notebook diagnoses why `release16_nonconvex_checkpoint_bc_official_replay_v1` failed the official warm-start replay and BC slice gate. It does not train or retain an actor.

Governance:
- Baselines: `release16_nonconvex_checkpoint_bc_official_replay_v1` and `soxlbacktest_qlib_walkforward_baseline_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `model_selection`, `target`, `bc_warm_start`.
- Acceptance: only a follow-up target contract can proceed; no runtime promotion from this diagnostic.
- Failure mode being tested: slice failure comes from missing/sparse target coverage, unstable state-head lift, or action-head mapping rather than a PPO length issue.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/soxl_quant").exists()
)
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.representation_diagnostics import (
    score_binary_representation_feature_sets,
    score_state_head_lift_stability,
)

ROOT = REPO_ROOT
CARD = ROOT / "runtime/model_selection/card/release16_sparse_slice_fit_root_cause_v1"
SOURCE_REPLAY = ROOT / "runtime/model_selection/card/release16_nonconvex_checkpoint_bc_official_replay_v1"
SOURCE_TARGET = ROOT / "runtime/model_selection/card/release16_failedfold_state_head_target_recipe_v1"
PRIOR_BC_SCREEN = ROOT / "runtime/model_selection/card/release16_failedfold_state_head_checkpoint_bc_screen_v1"

CARD.mkdir(parents=True, exist_ok=True)

summary = json.loads((SOURCE_REPLAY / "summary.json").read_text())
selection = json.loads((SOURCE_REPLAY / "selection.json").read_text())
bc_candidate_metrics = pd.read_csv(SOURCE_REPLAY / "bc_candidate_metrics.csv")
bc_slice_metrics = pd.read_csv(SOURCE_REPLAY / "bc_slice_metrics.csv")
target_coverage = pd.read_csv(SOURCE_REPLAY / "target_coverage.csv")
bc_action_distribution = pd.read_csv(SOURCE_REPLAY / "bc_action_distribution.csv")
drawdown_intervals = pd.read_csv(SOURCE_REPLAY / "drawdown_interval_diagnostic.csv")
target_contract = pd.read_csv(SOURCE_TARGET / "state_head_target_contract.csv", low_memory=False)
source_target_summary = json.loads((SOURCE_TARGET / "summary.json").read_text())

print(summary["decision"])
print(summary["next_mechanism"])
print(f"target rows: {len(target_contract):,}")


release16_nonconvex_checkpoint_bc_slice_failed_no_promotion
repair_sparse_slice_fit_or_state_head_feature_before_ppo
target rows: 3,731


In [2]:
# Build the selected-fold slice view and classify root cause per slice.
SLICE_THRESHOLDS = {
    "release_gate_state": {"min_samples": 20, "min_accuracy": 0.60},
    "is_rebound_cluster": {"min_samples": 20, "min_accuracy": 0.60},
    "drawdown_control_state": {"min_samples": 20, "min_accuracy": 0.60},
    "failedfold_state_head_target_state": {"min_samples": 5, "min_accuracy": 0.55},
    "is_high_participation_target": {"min_samples": 20, "min_accuracy": 0.60},
    "is_downtrend_state": {"min_samples": 50, "min_accuracy": 0.60},
    "all": {"min_samples": 1, "min_accuracy": 0.00},
}

selected_rows = []
for fold, payload in summary["selected_by_fold"].items():
    selected_rows.append(
        {
            "fold": fold,
            "selected_candidate": payload["selected_candidate"],
            "selected_relative_return": payload.get("selected_relative_return"),
            "selected_max_dd": payload.get("selected_max_dd"),
            "selected_avg_pos": payload.get("selected_avg_pos"),
            "selected_candidate_gate": payload.get("selected_candidate_gate"),
            "selected_slice_gate": payload.get("slice_gate"),
            "failed_slices_raw": payload.get("failed_slices", ""),
        }
    )
selected_by_fold = pd.DataFrame(selected_rows)

selected_slice_metrics = bc_slice_metrics.merge(selected_by_fold, on="fold", how="inner")
selected_slice_metrics = selected_slice_metrics[
    selected_slice_metrics["candidate"].eq(selected_slice_metrics["selected_candidate"])
].copy()

failed_slice_map = {
    row["fold"]: {item for item in str(row["failed_slices_raw"]).split(",") if item}
    for _, row in selected_by_fold.iterrows()
}

root_rows = []
for _, row in selected_slice_metrics.iterrows():
    slice_name = str(row["slice"])
    gate = SLICE_THRESHOLDS.get(slice_name, {"min_samples": 20, "min_accuracy": 0.60})
    samples = int(row["samples"]) if pd.notna(row["samples"]) else 0
    accuracy = float(row["accuracy"]) if pd.notna(row["accuracy"]) else np.nan
    pred_unique = int(row["predicted_unique_actions"]) if pd.notna(row["predicted_unique_actions"]) else 0
    pred_dom_share = float(row["predicted_dominant_share"]) if pd.notna(row["predicted_dominant_share"]) else np.nan
    action_mode_concentrated = bool(samples > 0 and (pred_unique <= 1 or pred_dom_share >= 0.90))
    if samples == 0:
        root_cause = "missing_target_slice_coverage"
        repair = "add_fold_local_target_rows_or_make_slice_gate_conditional_on_source_scope"
    elif samples < gate["min_samples"]:
        root_cause = "sparse_target_slice_under_min_samples"
        repair = "target_enrichment_or_aux_state_head_before_bc_replay"
    elif not np.isfinite(accuracy) or accuracy < gate["min_accuracy"]:
        root_cause = "bc_action_fit_accuracy_below_gate"
        repair = "state_conditioned_loss_or_aux_action_head_before_ppo"
    elif action_mode_concentrated:
        root_cause = "slice_passed_but_action_head_mode_concentrated"
        repair = "audit_action_mapping_and_add_state_conditioned_sampling"
    else:
        root_cause = "slice_fit_gate_passed"
        repair = "no_slice_repair_required"
    root_rows.append(
        {
            "fold": row["fold"],
            "candidate": row["candidate"],
            "slice": slice_name,
            "samples": samples,
            "accuracy": accuracy,
            "min_samples": int(gate["min_samples"]),
            "min_accuracy": float(gate["min_accuracy"]),
            "label_unique_actions": int(row["label_unique_actions"]),
            "predicted_unique_actions": pred_unique,
            "label_dominant_action": int(row["label_dominant_action"]),
            "label_dominant_share": float(row["label_dominant_share"]),
            "predicted_dominant_action": int(row["predicted_dominant_action"]),
            "predicted_dominant_share": pred_dom_share,
            "is_failed_in_selected_summary": slice_name in failed_slice_map.get(row["fold"], set()),
            "root_cause": root_cause,
            "repair_requirement": repair,
        }
    )

slice_failure_root_cause = pd.DataFrame(root_rows).sort_values(["fold", "slice"]).reset_index(drop=True)
slice_failure_root_cause.to_csv(CARD / "slice_failure_root_cause.csv", index=False)
slice_failure_root_cause


,fold,candidate,slice,samples,accuracy,min_samples,min_accuracy,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share,is_failed_in_selected_summary,root_cause,repair_requirement
0,wf_2020,risk_guarded,all,2156,0.629870,1,0.00,5,3,4,0.522727,4,0.693414,False,slice_fit_gate_passed,no_slice_repair_required
1,wf_2020,risk_guarded,drawdown_control_state,640,0.717187,20,0.60,5,3,4,0.518750,4,0.512500,False,slice_fit_gate_passed,no_slice_repair_required
2,wf_2020,risk_guarded,failedfold_state_head_target_state,1042,0.689060,5,0.55,5,3,4,0.668906,4,0.692898,False,slice_fit_gate_passed,no_slice_repair_required
3,wf_2020,risk_guarded,is_downtrend_state,640,0.717187,50,0.60,5,3,4,0.518750,4,0.512500,False,slice_fit_gate_passed,no_slice_repair_required
4,wf_2020,risk_guarded,is_high_participation_target,644,0.630435,20,0.60,5,3,4,0.664596,4,0.737578,False,slice_fit_gate_passed,no_slice_repair_required
5,wf_2020,risk_guarded,is_rebound_cluster,1071,0.690943,20,0.60,5,3,4,0.675070,4,0.694678,False,slice_fit_gate_passed,no_slice_repair_required
6,wf_2020,risk_guarded,release_gate_state,36,0.722222,20,0.60,3,3,4,0.833333,4,0.722222,False,slice_fit_gate_passed,no_slice_repair_required
7,wf_2022,risk_guarded,all,252,0.623016,1,0.00,5,2,0,0.420635,0,0.793651,False,slice_fit_gate_passed,no_slice_repair_required
8,wf_2022,risk_guarded,drawdown_control_state,0,NaN,20,0.60,0,0,-1,1.000000,-1,1.000000,True,missing_target_slice_coverage,add_fold_local_target_rows_or_make_slice_gate_...
9,wf_2022,risk_guarded,failedfold_state_head_target_state,0,NaN,5,0.55,0,0,-1,1.000000,-1,1.000000,True,missing_target_slice_coverage,add_fold_local_target_rows_or_make_slice_gate_...


In [3]:
# Target contract state coverage and action/lift distribution.
contract = target_contract.copy()
contract["datetime"] = pd.to_datetime(contract["datetime"])
if "fold_family" not in contract.columns:
    contract["fold_family"] = contract["fold"].astype(str).str.replace("_train", "", regex=False)

slice_cols = [
    "release_gate_state",
    "is_rebound_cluster",
    "drawdown_control_state",
    "failedfold_state_head_target_state",
    "is_high_participation_target",
    "is_downtrend_state",
]
for col in slice_cols:
    if col not in contract.columns:
        contract[col] = False
    contract[col] = contract[col].fillna(False).astype(bool)

for col in [
    "target_lift",
    "state_target_lift",
    "failedfold_state_head_target_lift",
    "risk_target_position",
    "risk_target_position_original",
    "rebound_target_position",
    "teacher_position",
    "objective_weight",
]:
    if col in contract.columns:
        contract[col] = pd.to_numeric(contract[col], errors="coerce").fillna(0.0)

coverage_rows = []
for fold, group in contract.groupby("fold_family", sort=True):
    for slice_name in slice_cols:
        mask = group[slice_name].fillna(False).astype(bool)
        threshold = SLICE_THRESHOLDS.get(slice_name, {"min_samples": 20, "min_accuracy": 0.60})
        subset = group[mask]
        if len(subset):
            action_counts = subset["action"].fillna("MISSING").astype(str).value_counts(normalize=True)
            dominant_action = str(action_counts.index[0])
            dominant_share = float(action_counts.iloc[0])
            max_lift = float(pd.to_numeric(subset.get("target_lift", 0.0), errors="coerce").fillna(0.0).max())
            state_head_lift_rows = int(
                (pd.to_numeric(subset.get("failedfold_state_head_target_lift", 0.0), errors="coerce").fillna(0.0).abs() > 0.01).sum()
            )
        else:
            dominant_action = ""
            dominant_share = np.nan
            max_lift = 0.0
            state_head_lift_rows = 0
        coverage_rows.append(
            {
                "fold": fold,
                "slice": slice_name,
                "contract_rows": int(len(group)),
                "slice_rows": int(len(subset)),
                "min_samples": int(threshold["min_samples"]),
                "sample_coverage_gate": bool(len(subset) >= int(threshold["min_samples"])),
                "dominant_action": dominant_action,
                "dominant_action_share": dominant_share,
                "unique_actions": int(subset["action"].nunique(dropna=True)) if len(subset) else 0,
                "max_target_lift": max_lift,
                "state_head_effective_lift_rows": state_head_lift_rows,
            }
        )

target_slice_coverage = pd.DataFrame(coverage_rows).sort_values(["fold", "slice"]).reset_index(drop=True)
target_slice_coverage.to_csv(CARD / "target_slice_coverage_by_fold.csv", index=False)

# Keep the original BC action distribution with selected-candidate marking.
action_distribution = bc_action_distribution.merge(selected_by_fold, on="fold", how="left")
action_distribution["is_selected_candidate"] = action_distribution["candidate"].eq(action_distribution["selected_candidate"])
action_distribution.to_csv(CARD / "bc_action_distribution_with_selection.csv", index=False)

target_slice_coverage


,fold,slice,contract_rows,slice_rows,min_samples,sample_coverage_gate,dominant_action,dominant_action_share,unique_actions,max_target_lift,state_head_effective_lift_rows
0,wf_2020,drawdown_control_state,2470,781,20,True,HOLD,0.966709,3,0.000000,781
1,wf_2020,failedfold_state_head_target_state,2470,1058,5,True,HOLD,0.981096,3,0.000000,1058
2,wf_2020,is_downtrend_state,2470,781,50,True,HOLD,0.966709,3,0.000000,781
3,wf_2020,is_high_participation_target,2470,749,20,True,HOLD,0.935915,3,0.000000,726
4,wf_2020,is_rebound_cluster,2470,1093,20,True,HOLD,0.954254,4,0.000000,1093
5,wf_2020,release_gate_state,2470,42,20,True,BUY,0.714286,2,0.000000,42
6,wf_2022,drawdown_control_state,505,0,20,False,,NaN,0,0.000000,0
7,wf_2022,failedfold_state_head_target_state,505,0,5,False,,NaN,0,0.000000,0
8,wf_2022,is_downtrend_state,505,234,50,True,MISSING,1.000000,0,0.302412,234
9,wf_2022,is_high_participation_target,505,34,20,True,MISSING,1.000000,0,0.000000,0


In [4]:
# Cross-fold feature separability for each target/slice label.
feature_sets = {
    "momentum_drawdown": ["mom_5", "mom_21", "drawdown_63"],
    "rulebase_reversal": [
        "RB_PANIC_SCORE_PROXY",
        "RB_DAILY_DROP_SEVERITY",
        "RB_CUMULATIVE_DROP_SEVERITY",
        "RB_RSI6_OVERSOLD",
        "RB_VOL_REGIME",
        "RB_BUY_COMPOSITE",
        "RB_SELL_COMPOSITE",
        "RB_PROFIT_TAKE_PROXY",
        "RB_VOLUME_STRESS",
        "RB_TREND_REVERSAL",
    ],
    "volatility_stress": ["realized_vol_21", "realized_vol_63", "vol_stress_score", "regime_risk", "rv_21"],
    "relative_strength_spread": [
        "relative_strength_21",
        "tracking_error_21",
        "tracking_error_vol_21",
        "soxl_soxx_ratio_widen_21",
        "soxl_soxx_ratio_z_252",
    ],
    "trend_quality": ["mom_63", "anchor_gap_180", "anchor_slope_180", "trend_gap_63", "decay_63"],
}
feature_sets = {
    name: [feature for feature in features if feature in contract.columns]
    for name, features in feature_sets.items()
}
feature_sets = {name: features for name, features in feature_sets.items() if features}

feature_score_frames = []
fold_metric_frames = []
feature_effect_frames = []
for label_col in slice_cols:
    result = score_binary_representation_feature_sets(
        contract,
        label_col=label_col,
        fold_col="fold_family",
        feature_sets=feature_sets,
        min_positive_train=1,
        min_positive_eval=1,
        min_holdout_auc=0.60,
        min_holdout_accuracy=0.55,
    )
    scores = result.feature_set_scores.copy()
    folds = result.fold_metrics.copy()
    effects = result.feature_effects.copy()
    scores.insert(0, "target_label", label_col)
    folds.insert(0, "target_label", label_col)
    effects.insert(0, "target_label", label_col)
    feature_score_frames.append(scores)
    fold_metric_frames.append(folds)
    feature_effect_frames.append(effects)

state_representation_feature_set_scores = pd.concat(feature_score_frames, ignore_index=True)
state_representation_fold_metrics = pd.concat(fold_metric_frames, ignore_index=True)
state_representation_feature_effects = pd.concat(feature_effect_frames, ignore_index=True)

state_representation_feature_set_scores.to_csv(CARD / "state_representation_feature_set_scores.csv", index=False)
state_representation_fold_metrics.to_csv(CARD / "state_representation_fold_metrics.csv", index=False)
state_representation_feature_effects.to_csv(CARD / "state_representation_feature_effects.csv", index=False)

best_representation_by_label = (
    state_representation_feature_set_scores.sort_values(
        ["target_label", "representation_gate", "holdout_auc_min", "holdout_accuracy_min"],
        ascending=[True, False, False, False],
    )
    .groupby("target_label", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_representation_by_label.to_csv(CARD / "best_representation_by_label.csv", index=False)
best_representation_by_label[
    ["target_label", "feature_set", "coverage_gate", "holdout_auc_min", "holdout_accuracy_min", "representation_gate"]
]


,target_label,feature_set,coverage_gate,holdout_auc_min,holdout_accuracy_min,representation_gate
0,drawdown_control_state,momentum_drawdown,False,NaN,NaN,False
1,failedfold_state_head_target_state,rulebase_reversal,False,0.875338,0.793651,False
2,is_downtrend_state,trend_quality,True,0.788895,0.683168,True
3,is_high_participation_target,momentum_drawdown,True,0.648355,0.429703,False
4,is_rebound_cluster,relative_strength_spread,True,0.757603,0.352475,False
5,release_gate_state,momentum_drawdown,True,0.689811,0.636243,True


In [5]:
# State-head lift stability audit: detect whether effective lift is fold-conflicting by date group.
stability_frame = contract.copy()
valid_features = pd.to_numeric(stability_frame.get("failedfold_state_head_valid_features", 0), errors="coerce").fillna(0)
feature_hits = pd.to_numeric(stability_frame.get("failedfold_state_head_feature_hits", 0), errors="coerce").fillna(0)
stability_frame["predicted_state_head_state"] = valid_features.ge(5) & feature_hits.ge(5)
if "date_key" not in stability_frame.columns:
    stability_frame["date_key"] = pd.to_datetime(stability_frame["datetime"]).dt.strftime("%Y-%m-%d")

state_head_lift_stability = score_state_head_lift_stability(
    stability_frame,
    group_col="date_key",
    label_col="failedfold_state_head_target_state",
    predicted_col="predicted_state_head_state",
    target_lift_col="failedfold_state_head_target_lift",
)
state_head_lift_stability.group_metrics.to_csv(CARD / "state_head_lift_stability_groups.csv", index=False)
state_head_lift_stability.row_scores.to_csv(CARD / "state_head_lift_stability_rows.csv", index=False)

pd.DataFrame([state_head_lift_stability.summary]).to_csv(CARD / "state_head_lift_stability_summary.csv", index=False)
state_head_lift_stability.summary


/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/src/soxl_quant/app/representation_diagnostics.py:376: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  work["label_unstable"] = work[group_col].map(unstable_by_group).fillna(False).astype(bool)


{'rows': 3731,
 'groups': 756,
 'unstable_groups': 213,
 'effective_lift_rows': 1528,
 'positive_effective_lift_rows': 1315,
 'false_positive_effective_lift_rows': 213,
 'unstable_effective_lift_rows': 426,
 'unstable_false_positive_effective_lift_rows': 213,
 'unstable_effective_lift_rate': 0.2787958115183246,
 'stability_gate': False}

In [6]:
# Compare this nonconvex BC official replay to the previous state-head checkpoint/BC screen when available.
comparison_rows = []
if (PRIOR_BC_SCREEN / "summary.json").exists():
    prior_summary = json.loads((PRIOR_BC_SCREEN / "summary.json").read_text())
    prior_by_fold = prior_summary.get("selected_by_fold", {})
    current_by_fold = summary.get("selected_by_fold", {})
    for fold in sorted(set(prior_by_fold) | set(current_by_fold)):
        p = prior_by_fold.get(fold, {})
        c = current_by_fold.get(fold, {})
        comparison_rows.append(
            {
                "fold": fold,
                "prior_candidate": p.get("selected_candidate"),
                "current_candidate": c.get("selected_candidate"),
                "prior_relative_return": p.get("selected_relative_return"),
                "current_relative_return": c.get("selected_relative_return"),
                "relative_delta_current_minus_prior": (
                    float(c.get("selected_relative_return")) - float(p.get("selected_relative_return"))
                    if c.get("selected_relative_return") is not None and p.get("selected_relative_return") is not None
                    else np.nan
                ),
                "prior_max_dd": p.get("selected_max_dd"),
                "current_max_dd": c.get("selected_max_dd"),
                "max_dd_delta_current_minus_prior": (
                    float(c.get("selected_max_dd")) - float(p.get("selected_max_dd"))
                    if c.get("selected_max_dd") is not None and p.get("selected_max_dd") is not None
                    else np.nan
                ),
                "prior_slice_gate": p.get("selected_bc_slice_gate"),
                "current_slice_gate": c.get("slice_gate"),
                "prior_candidate_gate": p.get("selected_candidate_gate"),
                "current_candidate_gate": c.get("selected_candidate_gate"),
            }
        )
prior_bc_screen_comparison = pd.DataFrame(comparison_rows)
prior_bc_screen_comparison.to_csv(CARD / "prior_bc_screen_comparison.csv", index=False)
prior_bc_screen_comparison


,fold,prior_candidate,current_candidate,prior_relative_return,current_relative_return,relative_delta_current_minus_prior,prior_max_dd,current_max_dd,max_dd_delta_current_minus_prior,prior_slice_gate,current_slice_gate,prior_candidate_gate,current_candidate_gate
0,wf_2020,bc_initialization,risk_guarded,27.840731,-40.584106,-68.424837,26.374299,35.850991,9.476692,False,True,False,False
1,wf_2022,bc_initialization,risk_guarded,72.363307,77.131669,4.768362,15.487774,17.626753,2.138979,False,False,False,False
2,wf_2023_2024,bc_initialization,release_heavy,-16.012653,20.611751,36.624405,30.559989,25.531369,-5.028620,False,False,False,False


In [7]:
# Repair handoff table. This is a requirement list, not a promoted target contract.
repair_rows = [
    {
        "repair_area": "drawdown_control_target_coverage",
        "affected_folds": "wf_2022,wf_2023_2024",
        "evidence": "drawdown_control_state has 0 target rows in both folds while official MaxDD remains above target in wf_2023_2024 and near target in wf_2022",
        "required_next_step": "build_fold_local_drawdown_control_rows_from_train-only drawdown intervals and merge into a joint target contract",
        "allowed_next": "target_contract_screen",
        "ppo_allowed": False,
    },
    {
        "repair_area": "release_rebound_sparse_labels",
        "affected_folds": "wf_2022,wf_2023_2024",
        "evidence": "release_gate_state rows are 9/11 in the source contract and selected BC slice samples are 8/11, below the 20-row gate",
        "required_next_step": "use auxiliary release/rebound state head or fold-local target enrichment before another BC warm-start replay",
        "allowed_next": "representation_or_target_semantics_screen",
        "ppo_allowed": False,
    },
    {
        "repair_area": "guard_fold_state_head_gate_semantics",
        "affected_folds": "wf_2022",
        "evidence": "failedfold_state_head_target_state has 0 rows because wf_2022 was the guard fold; this should be audited as source-scope coverage, not treated as a learned-slice failure by itself",
        "required_next_step": "make slice-gate requirements conditional on source target scope while retaining release/rebound/high-participation gates",
        "allowed_next": "bc_slice_gate_contract_fix",
        "ppo_allowed": False,
    },
    {
        "repair_area": "action_head_mapping",
        "affected_folds": "wf_2022,wf_2023_2024",
        "evidence": "selected candidates collapse several sparse slices to one dominant predicted action even when feature separability is available",
        "required_next_step": "add state-conditioned sampling/loss or auxiliary state/action head and require holdout sparse-slice action accuracy",
        "allowed_next": "bc_training_objective_screen",
        "ppo_allowed": False,
    },
    {
        "repair_area": "wf_2020_path_risk",
        "affected_folds": "wf_2020",
        "evidence": "slice gate passes, but official relative return is -40.58pp and MaxDD is 35.85%; this is path risk/trend capture, not sparse-slice coverage",
        "required_next_step": "pair any slice repair with drawdown-drift/checkpoint objective; do not evaluate PPO on slice readiness alone",
        "allowed_next": "checkpoint_or_reward_objective_screen",
        "ppo_allowed": False,
    },
]
repair_requirements = pd.DataFrame(repair_rows)
repair_requirements.to_csv(CARD / "repair_requirements.csv", index=False)
repair_requirements


,repair_area,affected_folds,evidence,required_next_step,allowed_next,ppo_allowed
0,drawdown_control_target_coverage,"wf_2022,wf_2023_2024",drawdown_control_state has 0 target rows in bo...,build_fold_local_drawdown_control_rows_from_tr...,target_contract_screen,False
1,release_rebound_sparse_labels,"wf_2022,wf_2023_2024",release_gate_state rows are 9/11 in the source...,use auxiliary release/rebound state head or fo...,representation_or_target_semantics_screen,False
2,guard_fold_state_head_gate_semantics,wf_2022,failedfold_state_head_target_state has 0 rows ...,make slice-gate requirements conditional on so...,bc_slice_gate_contract_fix,False
3,action_head_mapping,"wf_2022,wf_2023_2024",selected candidates collapse several sparse sl...,add state-conditioned sampling/loss or auxilia...,bc_training_objective_screen,False
4,wf_2020_path_risk,wf_2020,"slice gate passes, but official relative retur...",pair any slice repair with drawdown-drift/chec...,checkpoint_or_reward_objective_screen,False


In [8]:
# Write machine-readable selection and summary.
failed_selected_slices = slice_failure_root_cause[
    (slice_failure_root_cause["slice"] != "all")
    & (slice_failure_root_cause["root_cause"] != "slice_fit_gate_passed")
]
coverage_missing = slice_failure_root_cause[slice_failure_root_cause["root_cause"].eq("missing_target_slice_coverage")]
sparse_rows = slice_failure_root_cause[slice_failure_root_cause["root_cause"].eq("sparse_target_slice_under_min_samples")]
accuracy_fail = slice_failure_root_cause[slice_failure_root_cause["root_cause"].eq("bc_action_fit_accuracy_below_gate")]
mode_concentrated = slice_failure_root_cause[slice_failure_root_cause["root_cause"].eq("slice_passed_but_action_head_mode_concentrated")]
representation_pass_labels = best_representation_by_label[
    best_representation_by_label["representation_gate"].astype(bool)
]["target_label"].tolist()

CARD_REL = CARD.relative_to(ROOT)

def sidecar(name: str) -> str:
    return str(CARD_REL / name)

summary_payload = {
    "experiment": "release16_sparse_slice_fit_root_cause_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "role": "diagnose",
    "changed_layer": ["model_selection", "target", "bc_warm_start", "representation"],
    "decision": "release16_sparse_slice_target_repair_required_no_training_no_promotion",
    "primary_blocker": "sparse_slice_target_repair_required",
    "source_card": "release16_nonconvex_checkpoint_bc_official_replay_v1",
    "source_target_contract": "release16_failedfold_state_head_target_recipe_v1",
    "direct_comparison_baseline": "release16_failedfold_state_head_checkpoint_bc_screen_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "target_window": ["wf_2020", "wf_2022", "wf_2023_2024"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "bc_warm_start_allowed_next": False,
    "training_allowed_from_this_card": False,
    "target_revision_ready": False,
    "next_mechanism": "build_fold_local_joint_drawdown_release_targets_and_aux_state_head_before_bc_replay",
    "selected_gate_pass_count": int(summary.get("selected_gate_pass_count", 0)),
    "slice_gate_pass_count": int(summary.get("slice_gate_pass_count", 0)),
    "drift_gate_pass_count": int(summary.get("drift_gate_pass_count", 0)),
    "min_wf_relative_return_pp": summary.get("min_wf_relative_return_pp"),
    "worst_wf_max_drawdown_pct": summary.get("worst_wf_max_drawdown_pct"),
    "root_cause_counts": slice_failure_root_cause["root_cause"].value_counts().to_dict(),
    "failed_selected_slice_count": int(len(failed_selected_slices)),
    "missing_target_slice_count": int(len(coverage_missing)),
    "sparse_under_min_sample_slice_count": int(len(sparse_rows)),
    "accuracy_failed_slice_count": int(len(accuracy_fail)),
    "mode_concentrated_slice_count": int(len(mode_concentrated)),
    "representation_pass_labels": representation_pass_labels,
    "state_head_lift_stability": state_head_lift_stability.summary,
    "summary_points": [
        "Selected BC candidates still pass 0/3 official replay gates; no actor is retained.",
        "wf_2022 and wf_2023_2024 have zero drawdown_control_state target rows in the source contract.",
        "release_gate_state remains sparse in the later folds, with 8/11 selected BC samples versus a 20-sample slice gate.",
        "Release/downtrend labels are cross-fold separable in existing feature groups, so the blocker is target/action mapping rather than a pure feature-availability failure.",
        "The state-head lift audit fails stability because effective lift appears in fold-conflicting date groups; do not hand this directly to PPO.",
    ],
    "sidecars": {
        "slice_failure_root_cause": sidecar("slice_failure_root_cause.csv"),
        "target_slice_coverage_by_fold": sidecar("target_slice_coverage_by_fold.csv"),
        "bc_action_distribution_with_selection": sidecar("bc_action_distribution_with_selection.csv"),
        "state_representation_feature_set_scores": sidecar("state_representation_feature_set_scores.csv"),
        "state_representation_fold_metrics": sidecar("state_representation_fold_metrics.csv"),
        "state_representation_feature_effects": sidecar("state_representation_feature_effects.csv"),
        "best_representation_by_label": sidecar("best_representation_by_label.csv"),
        "state_head_lift_stability_groups": sidecar("state_head_lift_stability_groups.csv"),
        "state_head_lift_stability_rows": sidecar("state_head_lift_stability_rows.csv"),
        "state_head_lift_stability_summary": sidecar("state_head_lift_stability_summary.csv"),
        "prior_bc_screen_comparison": sidecar("prior_bc_screen_comparison.csv"),
        "repair_requirements": sidecar("repair_requirements.csv"),
        "selection": sidecar("selection.json"),
        "summary": sidecar("summary.json"),
        "conclusion": sidecar("conclusion.md"),
    },
}

selection_payload = {
    **summary_payload,
    "selected_by_fold": summary.get("selected_by_fold", {}),
    "handoff_requirements": repair_requirements.to_dict("records"),
    "acceptance_gate_for_next_run": {
        "fold_local_scope_required": True,
        "drawdown_control_rows_required_for_failed_folds": True,
        "release_rebound_min_samples": 20,
        "holdout_sparse_slice_accuracy_min": 0.60,
        "state_head_lift_stability_gate_required": True,
        "official_warm_start_replay_required": True,
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 20.0,
        "no_runtime_promotion_from_this_card": True,
    },
}

(CARD / "summary.json").write_text(json.dumps(summary_payload, indent=2, sort_keys=True) + "\n")
(CARD / "selection.json").write_text(json.dumps(selection_payload, indent=2, sort_keys=True) + "\n")

conclusion_lines = [
    "# release16 sparse slice fit root-cause diagnostic",
    "",
    "## Decision",
    "",
    f"`{summary_payload['decision']}`. No actor retention, no promotion, and no PPO handoff from this card.",
    "",
    "## Evidence",
    "",
    f"- Selected official replay gate count remains `{summary_payload['selected_gate_pass_count']}/3`; min relative return is `{summary_payload['min_wf_relative_return_pp']}` pp and worst MaxDD is `{summary_payload['worst_wf_max_drawdown_pct']}`%.",
    f"- Missing target slice coverage rows: `{summary_payload['missing_target_slice_count']}`; sparse under-sample rows: `{summary_payload['sparse_under_min_sample_slice_count']}`; action-fit failures: `{summary_payload['accuracy_failed_slice_count']}`.",
    "- `drawdown_control_state` has zero target rows in `wf_2022` and `wf_2023_2024`.",
    "- `release_gate_state` has only 8/11 selected BC samples in `wf_2022`/`wf_2023_2024`, below the 20-row gate.",
    f"- Existing feature groups can separate these labels for: `{', '.join(representation_pass_labels)}`.",
    f"- State-head lift stability gate: `{state_head_lift_stability.summary['stability_gate']}` with unstable effective lift rate `{state_head_lift_stability.summary['unstable_effective_lift_rate']:.3f}`.",
    "",
    "## Next Action",
    "",
    "Build a fold-local joint drawdown/release target contract and auxiliary state/action-head screen before another BC official replay. Keep the guard-fold state-head slice as source-scope coverage, not a standalone learned-slice failure, but retain release/rebound/high-participation gates.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n")

print(json.dumps({
    "decision": summary_payload["decision"],
    "next_mechanism": summary_payload["next_mechanism"],
    "missing_target_slice_count": summary_payload["missing_target_slice_count"],
    "sparse_under_min_sample_slice_count": summary_payload["sparse_under_min_sample_slice_count"],
    "accuracy_failed_slice_count": summary_payload["accuracy_failed_slice_count"],
    "mode_concentrated_slice_count": summary_payload["mode_concentrated_slice_count"],
    "state_head_lift_stability_gate": state_head_lift_stability.summary["stability_gate"],
}, indent=2))

{
  "decision": "release16_sparse_slice_target_repair_required_no_training_no_promotion",
  "next_mechanism": "build_fold_local_joint_drawdown_release_targets_and_aux_state_head_before_bc_replay",
  "missing_target_slice_count": 3,
  "sparse_under_min_sample_slice_count": 3,
  "accuracy_failed_slice_count": 2,
  "mode_concentrated_slice_count": 3,
  "state_head_lift_stability_gate": false
}


In [9]:
conclusion_text = (CARD / "conclusion.md").read_text()
display(Markdown(conclusion_text))


# release16 sparse slice fit root-cause diagnostic

## Decision

`release16_sparse_slice_target_repair_required_no_training_no_promotion`. No actor retention, no promotion, and no PPO handoff from this card.

## Evidence

- Selected official replay gate count remains `0/3`; min relative return is `-40.5841056381` pp and worst MaxDD is `35.85099103981264`%.
- Missing target slice coverage rows: `3`; sparse under-sample rows: `3`; action-fit failures: `2`.
- `drawdown_control_state` has zero target rows in `wf_2022` and `wf_2023_2024`.
- `release_gate_state` has only 8/11 selected BC samples in `wf_2022`/`wf_2023_2024`, below the 20-row gate.
- Existing feature groups can separate these labels for: `is_downtrend_state, release_gate_state`.
- State-head lift stability gate: `False` with unstable effective lift rate `0.279`.

## Next Action

Build a fold-local joint drawdown/release target contract and auxiliary state/action-head screen before another BC official replay. Keep the guard-fold state-head slice as source-scope coverage, not a standalone learned-slice failure, but retain release/rebound/high-participation gates.
